# cuda-engine — export validation (A100)

Proves the **export → install → use** loop works on real hardware.

The unit suite proves the package *files* are generated correctly, on CPU. It cannot prove the
package actually builds, loads, computes, and traces under `torch.compile` on a GPU. That is what
this notebook does.

**Cost:** Section A is free (reuses an existing run). Section B costs ~$0.10–0.30 for one cheap
kernel, and is only needed if you have no run directory available.

**Runtime:** Runtime → Change runtime type → **A100 GPU**.


## 0. Setup


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


In [ ]:
BRANCH = 'v2.2/torch-compat'

!git clone --branch {BRANCH} --depth 20 https://github.com/shivnarainms22/Cuda-Engine.git /content/Cuda-Engine
%cd /content/Cuda-Engine
!git log --oneline -3


In [ ]:
%pip install -q -e .

# Make the freshly installed package importable in this already-running kernel.
import site, importlib
site.main()
importlib.invalidate_caches()


In [ ]:
import cuda_engine, torch
print('cuda_engine', cuda_engine.__file__)
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


## A. Validate from an existing run (free)

Point `RUNS_ROOT` at any directory containing run folders — e.g. a prior eval on Drive.
Leave `RUN_ID` as `None` to auto-pick the newest run that has a polished kernel.


In [ ]:
from pathlib import Path

# e.g. '/content/drive/MyDrive/cuda-engine-evals/<some-eval>/runs'
RUNS_ROOT = Path.home() / '.cache' / 'cuda_engine' / 'runs'
RUN_ID = None   # or 'abc123def456'

def find_runs(root: Path):
    root = Path(root)
    if not root.is_dir():
        return []
    ok = []
    for d in root.iterdir():
        if (d / 'report.json').is_file() and (d / 'checkpoint.json').is_file():
            has_kernel = (d / 'stage5_polish' / 'final' / 'kernel.cu').is_file() or \
                         (d / 'stage2_codegen' / 'kernel.cu').is_file()
            if has_kernel and (d / 'inputs' / 'reference.py').is_file():
                ok.append(d)
    return sorted(ok, key=lambda p: p.stat().st_mtime, reverse=True)

candidates = find_runs(RUNS_ROOT)
print(f'{len(candidates)} exportable run(s) under {RUNS_ROOT}')
for d in candidates[:10]:
    print(' ', d.name)
if RUN_ID is None and candidates:
    RUN_ID = candidates[0].name
print('
RUN_ID =', RUN_ID)


### Mount Drive (only if your runs live there)


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# RUNS_ROOT = Path('/content/drive/MyDrive/cuda-engine-evals/<your-eval>')
# candidates = find_runs(RUNS_ROOT); RUN_ID = candidates[0].name if candidates else None
# print(RUN_ID)


## B. Or synthesize one cheap kernel (~$0.10–0.30)

Only run this if Section A found nothing. Budgets are deliberately tight — this exists to produce
*an artifact to export*, not to demonstrate synthesis.


In [ ]:
# import os
# from google.colab import userdata
# os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

# import torch
# from cuda_engine import synthesize, SynthesisConfig
# from cuda_engine.config import RetryBudgets

# def vector_add(a, b):
#     return a + b

# result = synthesize(
#     prompt='Generate a simple fp32 elementwise vector addition kernel over a 1-D tensor.',
#     reference=vector_add,
#     target='sm_80',
#     config=SynthesisConfig(
#         retry_budgets=RetryBudgets(codegen=2, performance=0),   # cheapest useful run
#         escalate_to_opus_on_bust=False,
#     ),
# )
# print('passed:', result.passed, '| run:', result.run_id)
# RUN_ID = result.run_id
# RUNS_ROOT = Path(result.artifacts_dir).parent


## C. The validation

Exports the run, `pip install`s it, then runs `standalone_check.py` in a **fresh subprocess whose
working directory is not the repo**, so `import cuda_engine` cannot resolve locally. If the package
needed the generator, this is where it would fail.


In [ ]:
assert RUN_ID, 'No RUN_ID — run Section A or B first.'

!python tools/export_validation/validate_export.py \
    --run-id {RUN_ID} --runs-root {RUNS_ROOT} --keep


### Interpreting the result

The **exit code** is the verdict — not the log text.

| Outcome | Meaning |
|---|---|
| `EXPORT VALIDATION PASSED` | export → install → build → run → `torch.compile` all work. The headline feature is real. |
| fails at `pip install` | packaging bug in the generated `pyproject.toml`. |
| fails at *kernel builds and runs* | the JIT `cpp_extension.load` path or the shipped `.so` is broken. |
| fails at *matches reference* | the exported kernel computes something different from what was verified — the most serious outcome. |
| fails at *fullgraph* | the fake/meta registration is not doing its job on real hardware. |
| fails at *control* | the comparison cannot detect a wrong answer, so the pass above it means nothing. |


In [ ]:
import subprocess
print('exit code:', subprocess.run(['python','tools/export_validation/validate_export.py',
                                    '--run-id', str(RUN_ID), '--runs-root', str(RUNS_ROOT)]).returncode)


## D. Record the evidence

Paste the output above into `docs/milestones/v2.2-export-evidence.md` and commit it. An unrecorded
green run is not evidence.
